In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

RUN_TAG = "random20k_wealth_resnet18"

IMAGES_DIR = "/content/drive/MyDrive/india_2015_images/gee_chips_2015_random20k"
CLUSTERS_CSV = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"

OUT_DIR = "/content/drive/MyDrive/models_india_2015"
os.makedirs(OUT_DIR, exist_ok=True)

print("IMAGES_DIR exists:", os.path.isdir(IMAGES_DIR), IMAGES_DIR)
print("CLUSTERS_CSV exists:", os.path.exists(CLUSTERS_CSV), CLUSTERS_CSV)
print("OUT_DIR:", OUT_DIR)

Mounted at /content/drive
IMAGES_DIR exists: True /content/drive/MyDrive/india_2015_images/gee_chips_2015_random20k
CLUSTERS_CSV exists: True /content/drive/MyDrive/IN2015_clusters_nightlights.csv
OUT_DIR: /content/drive/MyDrive/models_india_2015


In [3]:
# Scan image files from filenames
pat = re.compile(
    r"^c(?P<cluster>\d+)_s(?P<sample>\d+)_lat(?P<lat>-?\d+(?:\.\d+)?)_lon(?P<lon>-?\d+(?:\.\d+)?)\.png$"
)

rows = []

for fn in os.listdir(IMAGES_DIR):
    m = pat.match(fn)
    if m is None:
        continue  # skip non-matching files

    rows.append({
        "cluster_id": int(m.group("cluster")),
        "sample_id": int(m.group("sample")),
        "lat": float(m.group("lat")),
        "lon": float(m.group("lon")),
        "img_path": str(Path(IMAGES_DIR) / fn),
        "file": fn
    })

meta = pd.DataFrame(rows)
meta.head()

,cluster_id,sample_id,lat,lon,img_path,file
0,340405,1,30.05521,80.16280,/content/drive/MyDrive/india_2015_images/gee_c...,c340405_s1_lat30.05521_lon80.16280.png
1,340405,2,30.02609,80.22462,/content/drive/MyDrive/india_2015_images/gee_c...,c340405_s2_lat30.02609_lon80.22462.png
2,340405,3,30.03423,80.21097,/content/drive/MyDrive/india_2015_images/gee_c...,c340405_s3_lat30.03423_lon80.21097.png
3,340405,4,30.03942,80.15869,/content/drive/MyDrive/india_2015_images/gee_c...,c340405_s4_lat30.03942_lon80.15869.png
4,340405,5,30.10125,80.17767,/content/drive/MyDrive/india_2015_images/gee_c...,c340405_s5_lat30.10125_lon80.17767.png


## Merge cluster labels and make 3-class labels and group split

In [5]:
from sklearn.model_selection import GroupShuffleSplit

clusters = pd.read_csv(CLUSTERS_CSV)

assert "cluster_id" in clusters.columns
assert "target" in clusters.columns

# Merge image metadata with cluster targets
df = meta.merge(
    clusters[["cluster_id", "target"]],
    on="cluster_id",
    how="left"
)

df = df.dropna(subset=["target"]).reset_index(drop=True)

# Convert wealth target (1–5) into 3 classes: 1–2 = low, 3 = middle, 4–5 = high
df["label"] = pd.cut(
    df["target"],
    bins=[-np.inf, 2.5, 3.5, np.inf],
    labels=[0, 1, 2]
).astype(int)

# Split train/validation by cluster_id to avoid leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(df, groups=df["cluster_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
valid_df = df.iloc[valid_idx].reset_index(drop=True)

##  Dataloader

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

INPUT_SIZE = 224
BATCH_SIZE = 64

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ChipsDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y

train_ds = ChipsDataset(train_df, train_tfms)
valid_ds = ChipsDataset(valid_df, valid_tfms)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

INPUT_SIZE = 224
BATCH_SIZE = 64

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ChipsDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y

train_ds = ChipsDataset(train_df, train_tfms)
valid_ds = ChipsDataset(valid_df, valid_tfms)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


device: cuda


In [5]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

N_CLASSES = 3
MODEL_NAME = "resnet18"

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("Model ready:", MODEL_NAME)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 141MB/s]


Model ready: resnet18


In [6]:
from tqdm.auto import tqdm

EPOCHS = 8 if torch.cuda.is_available() else 3

best_acc = -1.0
best_path = os.path.join(OUT_DIR, f"cnn_{RUN_TAG}_best.pt")

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item()) * x.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += x.size(0)

    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(valid_loader, train=False)

    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "run_tag": RUN_TAG,
            "model_name": MODEL_NAME,
            "state_dict": model.state_dict(),
            "input_size": INPUT_SIZE,
            "n_classes": N_CLASSES,
            "best_val_acc": best_acc,
        }, best_path)

print("Best val acc:", best_acc)
print("Saved:", best_path)

  0%|          | 0/307 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

Epoch 1/8 | train loss 0.9227 acc 0.563 | val loss 1.0896 acc 0.525


  0%|          | 0/307 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

...

  0%|          | 0/78 [00:00<?, ?it/s]

Epoch 7/8 | train loss 0.7108 acc 0.687 | val loss 0.9958 acc 0.574


  0%|          | 0/307 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

Epoch 8/8 | train loss 0.6859 acc 0.699 | val loss 0.9697 acc 0.620
Best val acc: 0.6204453441295547
Saved: /content/drive/MyDrive/models_india_2015/cnn_random20k_wealth_resnet18_best.pt
